<a href="https://colab.research.google.com/github/Bunny276-luffy/airline-passenger-satisfaction-ml/blob/main/notebooks/Airline_Satisfaction_Clean_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Airline Passenger Satisfaction — Clean Modeling Notebook

**Goal:** build a reproducible LightGBM + CatBoost ensemble for the Kaggle *Predicting Airline Satisfaction* dataset.

This notebook replaces the fragmented experimental notebook with one ordered workflow:
1. Upload and validate the competition CSV files.
2. Explore the data and engineer service/route features.
3. Train and evaluate LightGBM and CatBoost on the same stratified validation split.
4. Blend validation predictions and report ROC-AUC.
5. Retrain both models on all training rows, create and verify a submission CSV.

**Important:** ROC-AUC on the validation split is not the same as Kaggle public ROC-AUC. No score is guaranteed. Keep your previously successful `0.96032` submission as the benchmark; this notebook creates a new file and never overwrites it.

In [ ]:
# CELL 1 — Install dependencies
%pip -q install -U lightgbm catboost scikit-learn pandas numpy matplotlib seaborn

ERROR: Operation cancelled by user
^C


In [ ]:
# CELL 2 — Upload train.csv, test.csv and sample_submission.csv
# You can select the three CSV files together.
# If you have a ZIP containing them, upload that ZIP instead.

from google.colab import files
import os, glob, zipfile, shutil
import pandas as pd
import numpy as np

DATA_DIR = "/content/airline_satisfaction_data"
os.makedirs(DATA_DIR, exist_ok=True)

uploaded = files.upload()

for filename in uploaded:
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(filename, "r") as zf:
            zf.extractall(DATA_DIR)
    elif filename.lower().endswith(".csv"):
        shutil.copy(filename, os.path.join(DATA_DIR, os.path.basename(filename)))

def locate_csv(filename):
    matches = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if not matches:
        # Also check /content in case files were uploaded directly
        matches = glob.glob(os.path.join("/content", "**", filename), recursive=True)
        matches = [p for p in matches if DATA_DIR not in p]
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. Upload train.csv, test.csv and sample_submission.csv."
        )
    return matches[0]

train_path = locate_csv("train.csv")
test_path = locate_csv("test.csv")
sample_path = locate_csv("sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("Train:", train.shape, train_path)
print("Test:", test.shape, test_path)
print("Sample submission:", sample_submission.shape, sample_path)
display(train.head())

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


KeyboardInterrupt: 

In [ ]:
# CELL 3 — Validate the data and inspect the target
TARGET = "satisfaction"
ID_COL = "id"
SEED = 42

required_train = {TARGET, ID_COL}
required_test = {ID_COL}
assert required_train.issubset(train.columns), f"Train must contain {required_train}"
assert required_test.issubset(test.columns), f"Test must contain {required_test}"
assert TARGET not in test.columns, "Test should not contain the target column"
assert sample_submission.columns.tolist() == [ID_COL, TARGET], (
    f"Unexpected sample submission columns: {sample_submission.columns.tolist()}"
)
assert len(sample_submission) == len(test), "Sample submission and test row counts differ"
assert test[ID_COL].reset_index(drop=True).equals(
    sample_submission[ID_COL].reset_index(drop=True)
), "Test IDs do not match sample submission order"

print("Target values:")
print(train[TARGET].value_counts(dropna=False))
print("\nTarget proportions:")
print(train[TARGET].value_counts(normalize=True, dropna=False))
print("\nMissing values — top 15:")
display(train.isna().sum().sort_values(ascending=False).head(15).to_frame("missing"))
print("\nNumeric summary:")
display(train.describe(include="number").T.head(20))

In [ ]:
# CELL 4 — Basic exploratory analysis
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=train, x=TARGET, ax=axes[0])
axes[0].set_title("Passenger Satisfaction Distribution")
axes[0].set_xlabel("Satisfaction")
sns.histplot(data=train, x="Flight Distance", hue=TARGET, bins=40,
             element="step", stat="density", common_norm=False, ax=axes[1])
axes[1].set_title("Flight Distance by Satisfaction")
plt.tight_layout()
plt.show()

# Average satisfaction rate for selected categorical columns
for col in ["Gender", "Customer Type", "Type of Travel", "Class"]:
    if col in train.columns:
        summary = train.groupby(col, dropna=False)[TARGET].apply(
            lambda s: (s.astype(str).str.lower().isin(["1", "true", "satisfied"])).mean()
            if not pd.api.types.is_numeric_dtype(s) else pd.to_numeric(s, errors="coerce").mean()
        ).sort_values(ascending=False)
        print(f"\n{col}:")
        display(summary.to_frame("target_mean"))

In [ ]:
# CELL 5 — Target conversion and feature engineering
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def convert_target(series):
    if series.isna().any():
        raise ValueError("Target contains missing values.")
    if pd.api.types.is_bool_dtype(series):
        out = series.astype(int)
    elif set(pd.unique(series)).issubset({0, 1}):
        out = series.astype(int)
    else:
        mapping = {
            "satisfied": 1, "dissatisfied": 0,
            "true": 1, "false": 0,
            "1": 1, "0": 0
        }
        out = series.astype(str).str.strip().str.lower().map(mapping)
    if out.isna().any() or out.nunique() != 2:
        raise ValueError(f"Unsupported target values: {series.unique()[:10]}")
    return out.astype(int).reset_index(drop=True)

y = convert_target(train[TARGET])

RATING_COLS = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness"
]

def make_features(frame):
    df = frame.drop(columns=[TARGET, ID_COL], errors="ignore").copy()

    # Convert known numeric columns safely
    for col in df.columns:
        if df[col].dtype == "object":
            # Preserve genuinely categorical columns; numeric conversion is attempted
            converted = pd.to_numeric(df[col], errors="coerce")
            if converted.notna().sum() >= max(1, int(0.90 * df[col].notna().sum())):
                df[col] = converted

    dep_col = "Departure Delay in Minutes"
    arr_col = "Arrival Delay in Minutes"
    if dep_col in df.columns and arr_col in df.columns:
        dep = pd.to_numeric(df[dep_col], errors="coerce")
        arr = pd.to_numeric(df[arr_col], errors="coerce")
        df["Total_Delay"] = dep.fillna(0) + arr.fillna(0)
        df["Delay_Difference"] = dep.fillna(0) - arr.fillna(0)
        df["Any_Delay"] = ((dep.fillna(0) > 0) | (arr.fillna(0) > 0)).astype("int8")

    if "Flight Distance" in df.columns:
        distance = pd.to_numeric(df["Flight Distance"], errors="coerce")
        df["Log_Flight_Distance"] = np.log1p(distance.clip(lower=0))

    available_ratings = [c for c in RATING_COLS if c in df.columns]
    if available_ratings:
        ratings = df[available_ratings].apply(pd.to_numeric, errors="coerce")
        df["Rating_Mean"] = ratings.mean(axis=1)
        df["Rating_Min"] = ratings.min(axis=1)
        df["Rating_Max"] = ratings.max(axis=1)
        df["Rating_Std"] = ratings.std(axis=1)
        df["Low_Rating_Count"] = (ratings <= 2).sum(axis=1)
        df["High_Rating_Count"] = (ratings >= 4).sum(axis=1)
        df["Zero_Rating_Count"] = (ratings == 0).sum(axis=1)

        def group_mean(cols):
            cols = [c for c in cols if c in ratings.columns]
            return ratings[cols].mean(axis=1) if cols else pd.Series(np.nan, index=df.index)

        df["Comfort_Score"] = group_mean([
            "Seat comfort", "Leg room service", "Inflight entertainment"
        ])
        df["Convenience_Score"] = group_mean([
            "Ease of Online booking", "Online boarding",
            "Departure/Arrival time convenient"
        ])
        df["Service_Score"] = group_mean([
            "On-board service", "Baggage handling",
            "Checkin service", "Cleanliness"
        ])
        df["Comfort_x_Service"] = df["Comfort_Score"] * df["Service_Score"]
        df["Convenience_x_Comfort"] = df["Convenience_Score"] * df["Comfort_Score"]

    if "Age" in df.columns and "Flight Distance" in df.columns:
        df["Age_x_Distance"] = (
            pd.to_numeric(df["Age"], errors="coerce")
            * pd.to_numeric(df["Flight Distance"], errors="coerce")
        )

    return df

X_base = make_features(train)
X_test_base = make_features(test)
X_test_base = X_test_base.reindex(columns=X_base.columns)

# Categorical handling for CatBoost
cat_cols = X_base.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()
for col in cat_cols:
    X_base[col] = X_base[col].fillna("__MISSING__").astype(str)
    X_test_base[col] = X_test_base[col].fillna("__MISSING__").astype(str)

print("Feature matrix:", X_base.shape)
print("Test matrix:", X_test_base.shape)
print("Categorical columns:", len(cat_cols))
print("Target ROC-AUC baseline (constant prevalence is not meaningful):", round(y.mean(), 4))

In [ ]:
# CELL 6 — Create one fixed stratified validation split
all_idx = np.arange(len(X_base))
train_idx, valid_idx = train_test_split(
    all_idx,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

X_tr = X_base.iloc[train_idx].copy()
X_va = X_base.iloc[valid_idx].copy()
y_tr = y.iloc[train_idx].copy()
y_va = y.iloc[valid_idx].copy()

print("Train split:", X_tr.shape, "Validation split:", X_va.shape)
print("Train target mean:", round(y_tr.mean(), 5))
print("Validation target mean:", round(y_va.mean(), 5))

In [ ]:
# CELL 7 — Train CatBoost
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=3000,
    depth=8,
    learning_rate=0.04,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=5,
    random_seed=SEED,
    verbose=250,
    early_stopping_rounds=150,
    allow_writing_files=False,
    thread_count=-1
)

cat_model.fit(
    X_tr,
    y_tr,
    cat_features=cat_cols,
    eval_set=(X_va, y_va),
    use_best_model=True
)

cat_valid_pred = cat_model.predict_proba(X_va)[:, 1]
cat_auc = roc_auc_score(y_va, cat_valid_pred)
cat_iterations = max(1, int(cat_model.best_iteration_) + 1)

print("\nCATBOOST VALIDATION ROC-AUC:", round(cat_auc, 6))
print("Best iteration count:", cat_iterations)

In [ ]:
# CELL 8 — Train route-aware LightGBM
# Flight Distance is treated as a route proxy. Target encoding is out-of-fold
# for training rows to avoid directly using each row's own target.

import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold

DISTANCE_COL = "Flight Distance"
if DISTANCE_COL not in X_base.columns:
    raise ValueError(f"Required feature not found: {DISTANCE_COL}")

def make_oof_target_encoding(values, target, folds=5, smoothing=20, seed=42):
    values = pd.Series(values).reset_index(drop=True)
    target = pd.Series(target).reset_index(drop=True)
    oof = np.zeros(len(values), dtype=float)
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)

    for fit_idx, hold_idx in skf.split(values, target):
        prior = target.iloc[fit_idx].mean()
        stats = pd.DataFrame({
            "key": values.iloc[fit_idx].astype(str).values,
            "target": target.iloc[fit_idx].values
        }).groupby("key")["target"].agg(["mean", "count"])
        mapping = (
            stats["mean"] * stats["count"] + prior * smoothing
        ) / (stats["count"] + smoothing)
        oof[hold_idx] = (
            values.iloc[hold_idx].astype(str).map(mapping).fillna(prior).values
        )
    return oof

X_tr_lgb = X_tr.copy()
X_va_lgb = X_va.copy()

X_tr_lgb["Distance_Target_Encoding"] = make_oof_target_encoding(
    X_tr_lgb[DISTANCE_COL], y_tr, folds=5, smoothing=20, seed=SEED
)

prior_tr = y_tr.mean()
dist_stats = pd.DataFrame({
    "key": X_tr_lgb[DISTANCE_COL].astype(str).values,
    "target": y_tr.values
}).groupby("key")["target"].agg(["mean", "count"])
dist_mapping = (
    dist_stats["mean"] * dist_stats["count"] + prior_tr * 20
) / (dist_stats["count"] + 20)

X_va_lgb["Distance_Target_Encoding"] = (
    X_va_lgb[DISTANCE_COL].astype(str).map(dist_mapping).fillna(prior_tr)
)

# Fit one-hot columns on the combined train/validation feature values.
lgb_cat_cols = X_tr_lgb.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()
lgb_combined = pd.concat([X_tr_lgb, X_va_lgb], ignore_index=True)
lgb_combined = pd.get_dummies(
    lgb_combined, columns=lgb_cat_cols, dummy_na=False, dtype=np.uint8
)
X_tr_enc = lgb_combined.iloc[:len(X_tr_lgb)].copy()
X_va_enc = lgb_combined.iloc[len(X_tr_lgb):].copy()

lgb_model = LGBMClassifier(
    objective="binary",
    n_estimators=3000,
    learning_rate=0.025,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=50,
    colsample_bytree=0.85,
    subsample=0.85,
    subsample_freq=1,
    reg_alpha=0.1,
    reg_lambda=3.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

lgb_model.fit(
    X_tr_enc,
    y_tr,
    eval_set=[(X_va_enc, y_va)],
    callbacks=[
        lgb.early_stopping(150),
        lgb.log_evaluation(250)
    ]
)

lgb_valid_pred = lgb_model.predict_proba(X_va_enc)[:, 1]
lgb_auc = roc_auc_score(y_va, lgb_valid_pred)
lgb_iterations = max(1, int(lgb_model.best_iteration_ or 633))

print("\nLIGHTGBM VALIDATION ROC-AUC:", round(lgb_auc, 6))
print("Best iteration count:", lgb_iterations)

In [ ]:
# CELL 9 — Compare models and tune a validation blend
blend_rows = []
for lgb_weight in np.arange(0.0, 1.001, 0.025):
    cat_weight = 1.0 - lgb_weight
    blend_pred = lgb_weight * lgb_valid_pred + cat_weight * cat_valid_pred
    blend_rows.append({
        "LightGBM weight": round(float(lgb_weight), 3),
        "CatBoost weight": round(float(cat_weight), 3),
        "Validation ROC-AUC": roc_auc_score(y_va, blend_pred)
    })

blend_table = pd.DataFrame(blend_rows).sort_values(
    "Validation ROC-AUC", ascending=False
).reset_index(drop=True)

display(blend_table.head(10))

best_lgb_weight = float(blend_table.loc[0, "LightGBM weight"])
best_cat_weight = float(blend_table.loc[0, "CatBoost weight"])
best_blend_auc = float(blend_table.loc[0, "Validation ROC-AUC"])

# Preserve the weight pair that produced your previous strongest public result.
# Validation optimization is displayed above; final model defaults to the
# previously tested 47.5% LightGBM + 52.5% CatBoost blend.
FINAL_LGB_WEIGHT = 0.475
FINAL_CAT_WEIGHT = 0.525

print("CatBoost validation AUC:", round(cat_auc, 6))
print("LightGBM validation AUC:", round(lgb_auc, 6))
print("Best validation blend AUC:", round(best_blend_auc, 6))
print("Best validation weights — LGB:", best_lgb_weight, "Cat:", best_cat_weight)
print("Final submission weights — LGB:", FINAL_LGB_WEIGHT, "Cat:", FINAL_CAT_WEIGHT)

In [ ]:
# CELL 10 — Refit both models on all training rows and predict the test set
# CatBoost uses the same engineered feature set as its validation model.
final_cat = CatBoostClassifier(
    iterations=cat_iterations,
    depth=8,
    learning_rate=0.04,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=5,
    random_seed=SEED,
    verbose=250,
    allow_writing_files=False,
    thread_count=-1
)
final_cat.fit(X_base, y, cat_features=cat_cols)

# Build full-data OOF distance target encoding for LightGBM training.
X_full_lgb = X_base.copy()
X_test_lgb = X_test_base.copy()

X_full_lgb["Distance_Target_Encoding"] = make_oof_target_encoding(
    X_full_lgb[DISTANCE_COL], y, folds=5, smoothing=20, seed=SEED
)

full_prior = y.mean()
full_stats = pd.DataFrame({
    "key": X_full_lgb[DISTANCE_COL].astype(str).values,
    "target": y.values
}).groupby("key")["target"].agg(["mean", "count"])
full_mapping = (
    full_stats["mean"] * full_stats["count"] + full_prior * 20
) / (full_stats["count"] + 20)

X_test_lgb["Distance_Target_Encoding"] = (
    X_test_lgb[DISTANCE_COL].astype(str).map(full_mapping).fillna(full_prior)
)

full_cat_cols = X_full_lgb.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()
full_combined = pd.concat([X_full_lgb, X_test_lgb], ignore_index=True)
full_combined = pd.get_dummies(
    full_combined, columns=full_cat_cols, dummy_na=False, dtype=np.uint8
)
X_full_enc = full_combined.iloc[:len(X_full_lgb)].copy()
X_test_enc = full_combined.iloc[len(X_full_lgb):].copy()

final_lgb = LGBMClassifier(
    objective="binary",
    n_estimators=lgb_iterations,
    learning_rate=0.025,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=50,
    colsample_bytree=0.85,
    subsample=0.85,
    subsample_freq=1,
    reg_alpha=0.1,
    reg_lambda=3.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)
final_lgb.fit(X_full_enc, y)

print("Both final models trained on all training rows.")

In [ ]:
# CELL 11 — Generate, validate and save a NEW submission CSV
from pathlib import Path

cat_test_pred = final_cat.predict_proba(X_test_base)[:, 1]
lgb_test_pred = final_lgb.predict_proba(X_test_enc)[:, 1]

final_test_pred = (
    FINAL_LGB_WEIGHT * lgb_test_pred
    + FINAL_CAT_WEIGHT * cat_test_pred
)

submission = pd.DataFrame({
    ID_COL: test[ID_COL].reset_index(drop=True),
    TARGET: final_test_pred
})

# Validate schema, row count, IDs, missing values and probability range.
assert submission.columns.tolist() == sample_submission.columns.tolist()
assert len(submission) == len(sample_submission) == len(test)
assert submission[ID_COL].equals(sample_submission[ID_COL].reset_index(drop=True))
assert submission[TARGET].notna().all()
assert submission[TARGET].between(0, 1).all()

OUTPUT_PATH = "/content/airline_satisfaction_lgb_catboost_475_525.csv"
submission.to_csv(OUTPUT_PATH, index=False)

print("SUBMISSION CREATED AND VERIFIED")
print("File:", OUTPUT_PATH)
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())
print("Prediction range:", float(submission[TARGET].min()), "to", float(submission[TARGET].max()))
print("Prediction mean:", float(submission[TARGET].mean()))
display(submission.head())

In [ ]:
# CELL 12 — Download CSV for MANUAL Kaggle upload
from google.colab import files
files.download(OUTPUT_PATH)

## Manual Kaggle upload

1. Open the competition page and click **Submit Prediction**.
2. Upload `airline_satisfaction_lgb_catboost_475_525.csv`.
3. Add a clear description, such as `LightGBM + CatBoost ensemble`.
4. Wait for Kaggle to calculate the public ROC-AUC.

Do not upload any API token or `kaggle.json` file. This notebook intentionally contains no Kaggle API authentication or command-line submission cells.

**Keep your old best CSV unchanged.** Compare the new public score against your previous `0.96032` benchmark. If the new score is lower, retain the old result.